# Preprocess the student LLM survey

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
input_file = next(iter(uploaded))
print(f'Uploaded: {input_file}')

In [ ]:
data = pd.read_excel(input_file)
original_rows = len(data)

# Clean column names and remove completely empty rows.
data.columns = [str(column).strip() for column in data.columns]
data = data.dropna(how='all').copy()

# Trim extra spaces and mark common missing-value text as missing.
missing_values = {'', 'na', 'n/a', 'nan', 'null', 'none', '-'}
def clean_text(value):
    if not isinstance(value, str):
        return value
    value = value.strip()
    if value.lower() in missing_values:
        return pd.NA
    return value

for column in data.select_dtypes(include='object').columns:
    data[column] = data[column].map(clean_text)

# Remove exact duplicate submissions.
duplicate_rows = int(data.duplicated().sum())
data = data.drop_duplicates().copy()

# Fill numeric missing values with the median. Leave text responses,
# including names, blank so anonymous responses stay anonymous.
for column in data.columns:
    if pd.api.types.is_numeric_dtype(data[column]) and data[column].notna().any():
        data[column] = data[column].fillna(data[column].median())

print(f'Rows before cleaning: {original_rows}')
print(f'Duplicate rows removed: {duplicate_rows}')
print('Remaining missing values by column:')
print(data.isna().sum().to_string())
data.head()

In [ ]:
output_file = 'cleaned_survey_data.xlsx'
data.to_excel(output_file, index=False)
files.download(output_file)